# APY Dataset — Data Cleaning

## Objective

This notebook investigates and handles data-quality issues in the historical Area, Production and Yield (APY) dataset for BharatCrop Intelligence.

The cleaning process will focus on:
- Validating column names, data types, and record structure
- Investigating missing and zero values
- Examining inconsistencies between Area, Production, and Yield
- Detecting suspicious values and potential outliers
- Identifying aggregate crop categories that could cause double-counting
- Saving a cleaned copy while preserving the original dataset

**Important:** Missing values will not automatically be replaced with zero, and unusual observations will not be removed without justification. Cleaning decisions must be supported by evidence.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent

apy_path = (
    project_root
    / "data"
    / "processed"
    / "historical_food_grains_apy_2013-14_to_2024-25.csv"
)

apy_df = pd.read_csv(apy_path)

print(f"Dataset loaded: {apy_path.name}")
print(f"Shape: {apy_df.shape}")
print(f"Original file exists: {apy_path.exists()}")

Dataset loaded: historical_food_grains_apy_2013-14_to_2024-25.csv
Shape: (31356, 40)
Original file exists: True


In [2]:
# Confirm the expected identifier columns and annual measures.
category_columns = ["State", "District", "Crop", "Season"]

year_columns = [
    col for col in apy_df.columns
    if col.startswith(("Area-", "Production-", "Yield-"))
]

expected_columns = set(category_columns + year_columns)
missing_expected = expected_columns - set(apy_df.columns)

print("Missing expected columns:", missing_expected)
print("Duplicate rows:", apy_df.duplicated().sum())
print(
    "Duplicate identifier combinations:",
    apy_df.duplicated(
        subset=category_columns
    ).sum()
)

print("\nData types:")
display(apy_df.dtypes.value_counts().rename("Column Count").to_frame())

Missing expected columns: set()
Duplicate rows: 0
Duplicate identifier combinations: 0

Data types:


,Column Count
float64,36
object,4


In [3]:
# Separate missing-value checks for identifiers and annual measures.
missing_report = pd.DataFrame({
    "Missing Count": apy_df.isna().sum(),
    "Missing (%)": (apy_df.isna().mean() * 100).round(2)
})

display(
    missing_report.loc[
        missing_report["Missing Count"] > 0
    ].sort_values("Missing (%)", ascending=False)
)

,Missing Count,Missing (%)
Production-2013-14,10548,33.64
Yield-2013-14,10548,33.64
Area-2013-14,10428,33.26
Production-2014-15,10152,32.38
Yield-2014-15,10152,32.38
Area-2014-15,10001,31.90
Production-2015-16,9778,31.18
Yield-2015-16,9778,31.18
Area-2015-16,9733,31.04
Production-2016-17,8929,28.48


In [5]:
# Find the 25 largest non-missing Yield values across all years.

import pandas as pd

# Rebuild the agricultural year list from the column names.
apy_years = sorted({
    col.split("-")[1] + "-" + col.split("-")[2]
    for col in apy_df.columns
    if col.startswith("Yield-")
})

extreme_yield_rows = []

for year in apy_years:
    area_col = f"Area-{year}"
    production_col = f"Production-{year}"
    yield_col = f"Yield-{year}"

    top_rows = apy_df.loc[
        apy_df[yield_col].notna(),
        [
            "State", "District", "Crop", "Season",
            area_col, production_col, yield_col
        ]
    ].nlargest(5, yield_col).copy()

    top_rows.columns = [
        "State", "District", "Crop", "Season",
        "Area", "Production", "Yield"
    ]
    top_rows.insert(0, "Year", year)

    extreme_yield_rows.append(top_rows)

extreme_yields = pd.concat(
    extreme_yield_rows,
    ignore_index=True
)

display(
    extreme_yields
    .sort_values("Yield", ascending=False)
    .head(25)
    .reset_index(drop=True)
)

,Year,State,District,Crop,Season,Area,Production,Yield
0,2023-24,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,0.00,0.00,385540.0
1,2023-24,Andaman And Nicobar Islands,South Andamans,Urad,Total,0.00,0.00,385540.0
2,2023-24,Andaman And Nicobar Islands,South Andamans,Total Pulses,Total,0.00,0.00,161579.0
3,2023-24,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,0.00,0.00,161579.0
4,2023-24,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,0.00,0.00,105588.0
5,2021-22,Madhya Pradesh,Narmadapuram,Other Pulses,Rabi,0.00,0.01,103571.0
6,2021-22,Madhya Pradesh,Narmadapuram,Other Pulses,Total,0.00,0.01,103571.0
7,2021-22,Madhya Pradesh,Harda,Other Pulses,Rabi,0.00,0.01,98889.0
8,2021-22,Madhya Pradesh,Harda,Other Pulses,Total,0.00,0.01,98889.0
9,2022-23,Goa,North Goa,Cereals,Kharif,0.01,0.47,44815.0


In [6]:
# Display the underlying values without rounding them to two decimals.

pd.set_option("display.precision", 10)

display(
    extreme_yields
    .sort_values("Yield", ascending=False)
    .head(25)
    .reset_index(drop=True)
)

# Restore a readable default for later notebook outputs.
pd.reset_option("display.precision")

,Year,State,District,Crop,Season,Area,Production,Yield
0,2023-24,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,0.00,0.00,385540.0
1,2023-24,Andaman And Nicobar Islands,South Andamans,Urad,Total,0.00,0.00,385540.0
2,2023-24,Andaman And Nicobar Islands,South Andamans,Total Pulses,Total,0.00,0.00,161579.0
3,2023-24,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,0.00,0.00,161579.0
4,2023-24,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,0.00,0.00,105588.0
5,2021-22,Madhya Pradesh,Narmadapuram,Other Pulses,Rabi,0.00,0.01,103571.0
6,2021-22,Madhya Pradesh,Narmadapuram,Other Pulses,Total,0.00,0.01,103571.0
7,2021-22,Madhya Pradesh,Harda,Other Pulses,Rabi,0.00,0.01,98889.0
8,2021-22,Madhya Pradesh,Harda,Other Pulses,Total,0.00,0.01,98889.0
9,2022-23,Goa,North Goa,Cereals,Kharif,0.01,0.47,44815.0


In [7]:
# Inspect the relationship between Area, Production, and Yield
# for the extreme records. Do not modify or remove any observations.

extreme_check = extreme_yields.sort_values(
    "Yield", ascending=False
).head(25).copy()

extreme_check["Area_is_zero"] = extreme_check["Area"].eq(0)
extreme_check["Production_is_zero"] = (
    extreme_check["Production"].eq(0)
)

display(
    extreme_check[
        [
            "Year", "State", "District", "Crop", "Season",
            "Area", "Production", "Yield",
            "Area_is_zero", "Production_is_zero"
        ]
    ]
)

,Year,State,District,Crop,Season,Area,Production,Yield,Area_is_zero,Production_is_zero
50,2023-24,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,0.00,0.00,385540.0,True,True
51,2023-24,Andaman And Nicobar Islands,South Andamans,Urad,Total,0.00,0.00,385540.0,True,True
53,2023-24,Andaman And Nicobar Islands,South Andamans,Total Pulses,Total,0.00,0.00,161579.0,True,True
52,2023-24,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,0.00,0.00,161579.0,True,True
54,2023-24,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,0.00,0.00,105588.0,True,True
40,2021-22,Madhya Pradesh,Narmadapuram,Other Pulses,Rabi,0.00,0.01,103571.0,True,False
41,2021-22,Madhya Pradesh,Narmadapuram,Other Pulses,Total,0.00,0.01,103571.0,True,False
42,2021-22,Madhya Pradesh,Harda,Other Pulses,Rabi,0.00,0.01,98889.0,True,False
43,2021-22,Madhya Pradesh,Harda,Other Pulses,Total,0.00,0.01,98889.0,True,False
45,2022-23,Goa,North Goa,Cereals,Kharif,0.01,0.47,44815.0,False,False


In [8]:
# Inspect the original APY records behind the most extreme Yield values.
# This avoids relying on the rounded values in the summary table.

extreme_keys = extreme_yields.sort_values(
    "Yield", ascending=False
).head(10)

detail_rows = []

for _, row in extreme_keys.iterrows():
    year = row["Year"]

    mask = (
        apy_df["State"].eq(row["State"])
        & apy_df["District"].eq(row["District"])
        & apy_df["Crop"].eq(row["Crop"])
        & apy_df["Season"].eq(row["Season"])
    )

    record = apy_df.loc[mask].iloc[0]

    detail_rows.append({
        "Year": year,
        "State": row["State"],
        "District": row["District"],
        "Crop": row["Crop"],
        "Season": row["Season"],
        "Area": record[f"Area-{year}"],
        "Production": record[f"Production-{year}"],
        "Yield": record[f"Yield-{year}"],
    })

detail_df = pd.DataFrame(detail_rows)

with pd.option_context("display.precision", 12):
    display(detail_df)

,Year,State,District,Crop,Season,Area,Production,Yield
0,2023-24,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,0.00,0.00,385540.0
1,2023-24,Andaman And Nicobar Islands,South Andamans,Urad,Total,0.00,0.00,385540.0
2,2023-24,Andaman And Nicobar Islands,South Andamans,Total Pulses,Total,0.00,0.00,161579.0
3,2023-24,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,0.00,0.00,161579.0
4,2023-24,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,0.00,0.00,105588.0
5,2021-22,Madhya Pradesh,Narmadapuram,Other Pulses,Rabi,0.00,0.01,103571.0
6,2021-22,Madhya Pradesh,Narmadapuram,Other Pulses,Total,0.00,0.01,103571.0
7,2021-22,Madhya Pradesh,Harda,Other Pulses,Rabi,0.00,0.01,98889.0
8,2021-22,Madhya Pradesh,Harda,Other Pulses,Total,0.00,0.01,98889.0
9,2022-23,Goa,North Goa,Cereals,Kharif,0.01,0.47,44815.0


In [9]:
# Inspect one extreme record without table rounding.

year = "2023-24"

mask = (
    apy_df["State"].eq("Andaman And Nicobar Islands")
    & apy_df["District"].eq("South Andamans")
    & apy_df["Crop"].eq("Urad")
    & apy_df["Season"].eq("Rabi")
)

record = apy_df.loc[mask].iloc[0]

print("Area:", repr(record[f"Area-{year}"]))
print("Production:", repr(record[f"Production-{year}"]))
print("Yield:", repr(record[f"Yield-{year}"]))

print("\nUnderlying values:")
print(f"Area       = {record[f'Area-{year}']:.12f}")
print(f"Production = {record[f'Production-{year}']:.12f}")
print(f"Yield      = {record[f'Yield-{year}']:.12f}")

Area: np.float64(0.0)
Production: np.float64(0.0)
Yield: np.float64(385540.0)

Underlying values:
Area       = 0.000000000000
Production = 0.000000000000
Yield      = 385540.000000000000


In [10]:
# Identify records with suspicious Area, Production, and Yield combinations.

suspicious_rows = []

for year in apy_years:
    area = apy_df[f"Area-{year}"]
    production = apy_df[f"Production-{year}"]
    yield_value = apy_df[f"Yield-{year}"]

    mask = (
        yield_value.gt(0)
        & (area.eq(0) | production.eq(0))
    )

    flagged = apy_df.loc[
        mask, ["State", "District", "Crop", "Season"]
    ].copy()

    flagged["Year"] = year
    flagged["Area"] = area.loc[mask].values
    flagged["Production"] = production.loc[mask].values
    flagged["Yield"] = yield_value.loc[mask].values

    suspicious_rows.append(flagged)

suspicious_yield_df = pd.concat(suspicious_rows, ignore_index=True)

print("Total suspicious records:", len(suspicious_yield_df))
print("\nRecords by year:")
display(
    suspicious_yield_df.groupby("Year").size()
    .rename("Flagged Records")
    .to_frame()
)

print("\nHighest-yield flagged records:")
display(
    suspicious_yield_df.nlargest(15, "Yield").reset_index(drop=True)
)

Total suspicious records: 94235

Records by year:


,Flagged Records
Year,
2013-14,6468
2014-15,6187
2015-16,6680
2016-17,7008
2017-18,7316
2018-19,8024
2019-20,8397
2020-21,8585
2021-22,8424



Highest-yield flagged records:


,State,District,Crop,Season,Year,Area,Production,Yield
0,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,2023-24,0.0,0.00,385540.0
1,Andaman And Nicobar Islands,South Andamans,Urad,Total,2023-24,0.0,0.00,385540.0
2,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,2023-24,0.0,0.00,161579.0
3,Andaman And Nicobar Islands,South Andamans,Total Pulses,Total,2023-24,0.0,0.00,161579.0
4,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,2023-24,0.0,0.00,105588.0
5,Andaman And Nicobar Islands,South Andamans,Moong,Total,2023-24,0.0,0.00,105588.0
6,Madhya Pradesh,Narmadapuram,Other Pulses,Rabi,2021-22,0.0,0.01,103571.0
7,Madhya Pradesh,Narmadapuram,Other Pulses,Total,2021-22,0.0,0.01,103571.0
8,Madhya Pradesh,Harda,Other Pulses,Rabi,2021-22,0.0,0.01,98889.0
9,Madhya Pradesh,Harda,Other Pulses,Total,2021-22,0.0,0.01,98889.0


In [11]:
# Classify positive-yield records by Area and Production status.

classification = pd.DataFrame(index=apy_df.index)

for year in apy_years:
    area = apy_df[f"Area-{year}"]
    production = apy_df[f"Production-{year}"]
    yield_value = apy_df[f"Yield-{year}"]

    positive_yield = yield_value.gt(0)

    conditions = [
        positive_yield & area.eq(0) & production.eq(0),
        positive_yield & area.eq(0) & production.gt(0),
        positive_yield & area.gt(0) & production.eq(0),
        positive_yield & (area.isna() | production.isna()),
        positive_yield & area.gt(0) & production.gt(0),
    ]

    labels = [
        "Area=0 and Production=0",
        "Area=0 and Production>0",
        "Area>0 and Production=0",
        "Area or Production missing",
        "Area>0 and Production>0",
    ]

    year_result = pd.Series(
        np.select(conditions, labels, default="Other"),
        index=apy_df.index,
        name=year,
    )

    classification[year] = year_result

summary = classification.apply(
    lambda col: col.value_counts()
).fillna(0).astype(int)

display(summary)

,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25
Area=0 and Production=0,5296,5068,5338,5768,6011,6700,6993,7193,7025,7471,7805,7682
Area=0 and Production>0,602,554,593,655,784,769,837,908,896,918,921,921
Area>0 and Production=0,570,565,749,585,521,555,567,484,503,536,451,441
Area>0 and Production>0,14340,14981,14859,15416,15998,15468,15757,15751,15544,16088,16258,16127
Other,10548,10188,9817,8932,8042,7864,7202,7020,7388,6343,5921,6185


In [12]:
# Break down records with missing or non-positive Yield.

yield_status_rows = []

for year in apy_years:
    area = apy_df[f"Area-{year}"]
    production = apy_df[f"Production-{year}"]
    yield_value = apy_df[f"Yield-{year}"]

    status = np.select(
        [
            yield_value.isna(),
            yield_value.eq(0),
            yield_value.lt(0),
            yield_value.gt(0),
        ],
        [
            "Yield missing",
            "Yield = 0",
            "Yield < 0",
            "Yield > 0",
        ],
        default="Unclassified",
    )

    temp = pd.DataFrame({
        "Year": year,
        "Yield Status": status,
        "Area Missing": area.isna(),
        "Production Missing": production.isna(),
        "Area Zero": area.eq(0),
        "Production Zero": production.eq(0),
    })

    yield_status_rows.append(temp)

yield_status_df = pd.concat(yield_status_rows, ignore_index=True)

display(
    yield_status_df.groupby(
        ["Yield Status", "Area Missing", "Production Missing"],
        dropna=False
    ).size().rename("Record Count").reset_index()
    .sort_values("Record Count", ascending=False)
)

,Yield Status,Area Missing,Production Missing,Record Count
1,Yield > 0,False,False,280822
4,Yield missing,True,True,94180
2,Yield missing,False,True,850
0,Yield = 0,False,False,417
3,Yield missing,True,False,3


In [13]:
# Create a readable summary of Yield status and missing measurements.

summary_rows = []

for year in apy_years:
    area = apy_df[f"Area-{year}"]
    production = apy_df[f"Production-{year}"]
    yield_value = apy_df[f"Yield-{year}"]

    summary_rows.append({
        "Year": year,
        "Yield Missing": int(yield_value.isna().sum()),
        "Yield Zero": int(yield_value.eq(0).sum()),
        "Yield Negative": int(yield_value.lt(0).sum()),
        "Yield Positive": int(yield_value.gt(0).sum()),
        "Area Missing": int(area.isna().sum()),
        "Production Missing": int(production.isna().sum()),
        "All Three Missing": int(
            (area.isna() & production.isna() & yield_value.isna()).sum()
        ),
    })

yield_summary = pd.DataFrame(summary_rows)

display(yield_summary)

,Year,Yield Missing,Yield Zero,Yield Negative,Yield Positive,Area Missing,Production Missing,All Three Missing
0,2013-14,10548,0,0,20808,10428,10548,10428
1,2014-15,10152,36,0,21168,10001,10152,10001
2,2015-16,9778,39,0,21539,9733,9778,9733
3,2016-17,8929,3,0,22424,8867,8929,8867
4,2017-18,8036,6,0,23314,7970,8036,7970
5,2018-19,7864,0,0,23492,7743,7864,7743
6,2019-20,7196,6,0,24154,7052,7196,7052
7,2020-21,6965,55,0,24336,6965,6965,6965
8,2021-22,7331,57,0,23968,7331,7331,7331
9,2022-23,6173,170,0,25013,6173,6173,6173


In [14]:
# APY data-quality summary: one row per year, one column per metric.

summary_rows = []

for year in apy_years:
    area = apy_df[f"Area-{year}"]
    production = apy_df[f"Production-{year}"]
    yield_value = apy_df[f"Yield-{year}"]

    summary_rows.append({
        "Year": year,
        "Yield_missing": yield_value.isna().sum(),
        "Yield_zero": yield_value.eq(0).sum(),
        "Yield_positive": yield_value.gt(0).sum(),
        "Area_missing": area.isna().sum(),
        "Production_missing": production.isna().sum(),
        "All_three_missing": (
            area.isna() & production.isna() & yield_value.isna()
        ).sum(),
        "Area_zero_Production_positive": (
            area.eq(0) & production.gt(0)
        ).sum(),
        "Area_positive_Production_zero": (
            area.gt(0) & production.eq(0)
        ).sum(),
    })

quality_report = pd.DataFrame(summary_rows)

print("Rows per year:", len(quality_report))
print("Expected rows:", len(apy_years))
print("\nData-quality report:")
display(quality_report.set_index("Year").T)

Rows per year: 12
Expected rows: 12

Data-quality report:


Year,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25
Yield_missing,10548,10152,9778,8929,8036,7864,7196,6965,7331,6173,5897,6164
Yield_zero,0,36,39,3,6,0,6,55,57,170,24,21
Yield_positive,20808,21168,21539,22424,23314,23492,24154,24336,23968,25013,25435,25171
Area_missing,10428,10001,9733,8867,7970,7743,7052,6965,7331,6173,5803,6117
Production_missing,10548,10152,9778,8929,8036,7864,7196,6965,7331,6173,5894,6164
All_three_missing,10428,10001,9733,8867,7970,7743,7052,6965,7331,6173,5800,6117
Area_zero_Production_positive,602,554,593,655,784,769,837,908,896,918,921,921
Area_positive_Production_zero,570,593,753,585,521,555,568,484,508,545,455,445


In [15]:
# Check how many records have missing measurements across all years.

measurement_columns = [
    col for col in apy_df.columns
    if col.startswith(("Area-", "Production-", "Yield-"))
]

# Count missing measurement values per record.
apy_df["Missing_Measurement_Count"] = (
    apy_df[measurement_columns].isna().sum(axis=1)
)

print("Missing measurement values per record:")
display(
    apy_df["Missing_Measurement_Count"]
    .value_counts()
    .sort_index()
    .rename_axis("Missing Values Across 36 Measurements")
    .to_frame("Record Count")
)

# Check records where all three measurements are missing in every year.
all_missing_years = []

for year in apy_years:
    all_missing = (
        apy_df[f"Area-{year}"].isna()
        & apy_df[f"Production-{year}"].isna()
        & apy_df[f"Yield-{year}"].isna()
    )
    all_missing_years.append(all_missing.rename(year))

all_missing_matrix = pd.concat(all_missing_years, axis=1)

print("\nRecords with all three measurements missing in every year:",
      all_missing_matrix.all(axis=1).sum())

print("Records with all three measurements missing in at least one year:",
      all_missing_matrix.any(axis=1).sum())

Missing measurement values per record:


,Record Count
Missing Values Across 36 Measurements,
0,15920
2,83
3,2032
4,4
5,45
6,1225
7,8
8,38
9,1836



Records with all three measurements missing in every year: 0
Records with all three measurements missing in at least one year: 15348


In [16]:
# Measure historical coverage for each APY record.

coverage_rows = []

for year in apy_years:
    area = apy_df[f"Area-{year}"]
    production = apy_df[f"Production-{year}"]
    yield_value = apy_df[f"Yield-{year}"]

    coverage_rows.append(
        pd.DataFrame({
            "Year": year,
            "Area_Available": area.notna(),
            "Production_Available": production.notna(),
            "Yield_Available": yield_value.notna(),
            "All_Three_Available": (
                area.notna() & production.notna() & yield_value.notna()
            ),
        })
    )

coverage_df = pd.concat(coverage_rows, ignore_index=True)

# Summarize coverage at the individual-record level.
coverage_per_record = pd.DataFrame({
    "Area_Years_Available": apy_df[
        [f"Area-{year}" for year in apy_years]
    ].notna().sum(axis=1),
    "Production_Years_Available": apy_df[
        [f"Production-{year}" for year in apy_years]
    ].notna().sum(axis=1),
    "Yield_Years_Available": apy_df[
        [f"Yield-{year}" for year in apy_years]
    ].notna().sum(axis=1),
})

coverage_per_record["Complete_Years"] = pd.concat(
    [
        (
            apy_df[f"Area-{year}"].notna()
            & apy_df[f"Production-{year}"].notna()
            & apy_df[f"Yield-{year}"].notna()
        ).rename(year)
        for year in apy_years
    ],
    axis=1,
).sum(axis=1)

print("Complete-year coverage per record:")
display(
    coverage_per_record["Complete_Years"]
    .value_counts()
    .sort_index()
    .rename_axis("Years With All Three Measurements")
    .to_frame("Record Count")
)

print("\nCoverage summary:")
display(coverage_per_record.describe().round(2))

Complete-year coverage per record:


,Record Count
Years With All Three Measurements,
0,92
1,2485
2,1560
3,1611
4,742
5,1035
6,857
7,726
8,1061



Coverage summary:


,Area_Years_Available,Production_Years_Available,Yield_Years_Available,Complete_Years
count,31356.00,31356.00,31356.00,31356.00
mean,9.00,8.97,8.97,8.97
std,3.97,4.00,4.00,4.00
min,0.00,0.00,0.00,0.00
25%,6.00,6.00,6.00,6.00
50%,12.00,12.00,12.00,12.00
75%,12.00,12.00,12.00,12.00
max,12.00,12.00,12.00,12.00


In [17]:
# Count Area–Production consistency cases for every year.

consistency_rows = []

for year in apy_years:
    area = apy_df[f"Area-{year}"]
    production = apy_df[f"Production-{year}"]

    consistency_rows.append({
        "Year": year,
        "Both_Missing": (area.isna() & production.isna()).sum(),
        "Area_Missing_Only": (area.isna() & production.notna()).sum(),
        "Production_Missing_Only": (area.notna() & production.isna()).sum(),
        "Both_Zero": (area.eq(0) & production.eq(0)).sum(),
        "Area_Zero_Production_Positive": (
            area.eq(0) & production.gt(0)
        ).sum(),
        "Area_Positive_Production_Zero": (
            area.gt(0) & production.eq(0)
        ).sum(),
        "Both_Positive": (area.gt(0) & production.gt(0)).sum(),
        "Any_Negative": (area.lt(0) | production.lt(0)).sum(),
    })

consistency_report = pd.DataFrame(consistency_rows).set_index("Year")

display(consistency_report.T)

Year,2013-14,2014-15,2015-16,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25
Both_Missing,10428,10001,9733,8867,7970,7743,7052,6965,7331,6173,5800,6117
Area_Missing_Only,0,0,0,0,0,0,0,0,0,0,3,0
Production_Missing_Only,120,151,45,62,66,121,144,0,0,0,94,47
Both_Zero,5296,5076,5373,5771,6017,6700,6998,7248,7077,7632,7825,7699
Area_Zero_Production_Positive,602,554,593,655,784,769,837,908,896,918,921,921
Area_Positive_Production_Zero,570,593,753,585,521,555,568,484,508,545,455,445
Both_Positive,14340,14981,14859,15416,15998,15468,15757,15751,15544,16088,16258,16127
Any_Negative,0,0,0,0,0,0,0,0,0,0,0,0


In [18]:
# Create explicit data-quality flags without modifying source measurements.

apy_clean_df = apy_df.copy()

for year in apy_years:
    area_col = f"Area-{year}"
    production_col = f"Production-{year}"
    yield_col = f"Yield-{year}"

    area = apy_clean_df[area_col]
    production = apy_clean_df[production_col]
    yield_value = apy_clean_df[yield_col]

    apy_clean_df[f"QC-{year}-AreaZero-ProductionPositive"] = (
        area.eq(0) & production.gt(0)
    )

    apy_clean_df[f"QC-{year}-AreaPositive-ProductionZero"] = (
        area.gt(0) & production.eq(0)
    )

    apy_clean_df[f"QC-{year}-NegativeMeasurements"] = (
        area.lt(0) | production.lt(0) | yield_value.lt(0)
    )

    apy_clean_df[f"QC-{year}-PositiveYieldWithoutPositiveAreaProduction"] = (
        yield_value.gt(0)
        & ~(area.gt(0) & production.gt(0))
    )

print("Original shape:", apy_df.shape)
print("Flagged dataset shape:", apy_clean_df.shape)

print("\n2024-25 quality-flag counts:")
qc_columns = [
    col for col in apy_clean_df.columns
    if col.startswith("QC-2024-25-")
]

display(
    apy_clean_df[qc_columns]
    .sum()
    .rename("Flagged Records")
    .to_frame()
)

Original shape: (31356, 41)
Flagged dataset shape: (31356, 89)

2024-25 quality-flag counts:


,Flagged Records
QC-2024-25-AreaZero-ProductionPositive,921
QC-2024-25-AreaPositive-ProductionZero,445
QC-2024-25-NegativeMeasurements,0
QC-2024-25-PositiveYieldWithoutPositiveAreaProduction,9044


In [19]:
# Inspect crop categories and identify potential aggregate rows.

crop_summary = (
    apy_clean_df["Crop"]
    .value_counts(dropna=False)
    .rename_axis("Crop")
    .reset_index(name="Record Count")
)

aggregate_keywords = [
    "total", "cereals", "pulses", "food grains", "nutri/coarse"
]

crop_summary["Potential Aggregate"] = crop_summary["Crop"].str.lower().apply(
    lambda crop: any(keyword in crop for keyword in aggregate_keywords)
    if isinstance(crop, str) else False
)

print("Unique crop labels:", apy_clean_df["Crop"].nunique())
print("\nPotential aggregate categories:")
display(
    crop_summary[crop_summary["Potential Aggregate"]]
    .sort_values("Record Count", ascending=False)
)

print("\nAll crop categories:")
display(crop_summary)

Unique crop labels: 19

Potential aggregate categories:


,Crop,Record Count,Potential Aggregate
0,Total Food Grains,2759,True
1,Cereals,2677,True
2,Nutri/Coarse Cereals,2530,True
6,Other Pulses,1960,True
8,Shree Anna /Nutri Cereals,1803,True
18,Total Pulses,747,True



All crop categories:


,Crop,Record Count,Potential Aggregate
0,Total Food Grains,2759,True
1,Cereals,2677,True
2,Nutri/Coarse Cereals,2530,True
3,Maize,2317,False
4,Rice,2138,False
5,Moong,2074,False
6,Other Pulses,1960,True
7,Urad,1928,False
8,Shree Anna /Nutri Cereals,1803,True
9,Tur,1454,False


In [20]:
# Classify crop labels conservatively.
# This is an initial analytical classification, not a verified source taxonomy.

def classify_crop(crop):
    crop_normalized = str(crop).strip().lower()

    if crop_normalized.startswith("total "):
        return "Aggregate Total — Exclude from Individual-Crop Analysis"

    if crop_normalized in {
        "cereals",
        "nutri/coarse cereals",
        "shree anna /nutri cereals",
        "other pulses",
    }:
        return "Grouping — Verify Source Definition"

    return "Individual Crop Candidate"

crop_classification = (
    apy_clean_df[["Crop"]]
    .drop_duplicates()
    .assign(
        Crop_Category=lambda df: df["Crop"].apply(classify_crop)
    )
    .sort_values(["Crop_Category", "Crop"])
    .reset_index(drop=True)
)

display(crop_classification)

print("\nClassification counts:")
display(
    crop_classification["Crop_Category"]
    .value_counts()
    .rename("Number of Crop Labels")
    .to_frame()
)

,Crop,Crop_Category
0,Total Food Grains,Aggregate Total — Exclude from Individual-Crop...
1,Total Pulses,Aggregate Total — Exclude from Individual-Crop...
2,Cereals,Grouping — Verify Source Definition
3,Nutri/Coarse Cereals,Grouping — Verify Source Definition
4,Other Pulses,Grouping — Verify Source Definition
5,Shree Anna /Nutri Cereals,Grouping — Verify Source Definition
6,Bajra,Individual Crop Candidate
7,Barley,Individual Crop Candidate
8,Gram,Individual Crop Candidate
9,Jowar,Individual Crop Candidate



Classification counts:


,Number of Crop Labels
Crop_Category,
Individual Crop Candidate,13
Grouping — Verify Source Definition,4
Aggregate Total — Exclude from Individual-Crop Analysis,2


In [21]:
# Prepare the cleaned APY dataset.
# Preserve source measurements and retain quality flags for transparency.

# Start from the original APY data, excluding temporary analysis columns.
apy_clean_df = apy_df.drop(
    columns=["Missing_Measurement_Count"],
    errors="ignore"
).copy()

# Add provisional crop classifications.
crop_category_map = dict(
    zip(crop_classification["Crop"], crop_classification["Crop_Category"])
)

apy_clean_df["Crop_Category"] = apy_clean_df["Crop"].map(crop_category_map)

# Add quality flags for each year.
for year in apy_years:
    area = apy_clean_df[f"Area-{year}"]
    production = apy_clean_df[f"Production-{year}"]
    yield_value = apy_clean_df[f"Yield-{year}"]

    apy_clean_df[f"QC-{year}-AreaZero-ProductionPositive"] = (
        area.eq(0) & production.gt(0)
    )

    apy_clean_df[f"QC-{year}-AreaPositive-ProductionZero"] = (
        area.gt(0) & production.eq(0)
    )

    apy_clean_df[f"QC-{year}-NegativeMeasurements"] = (
        area.lt(0) | production.lt(0) | yield_value.lt(0)
    )

    apy_clean_df[f"QC-{year}-PositiveYieldWithoutPositiveAreaProduction"] = (
        yield_value.gt(0)
        & ~(area.gt(0) & production.gt(0))
    )

print("Original source shape:", apy_df.drop(
    columns=["Missing_Measurement_Count"], errors="ignore"
).shape)
print("Prepared dataset shape:", apy_clean_df.shape)
print("Duplicate rows:", apy_clean_df.duplicated().sum())
print("Missing crop classifications:", apy_clean_df["Crop_Category"].isna().sum())

display(apy_clean_df["Crop_Category"].value_counts().to_frame("Record Count"))

Original source shape: (31356, 40)
Prepared dataset shape: (31356, 89)
Duplicate rows: 0
Missing crop classifications: 0


,Record Count
Crop_Category,
Individual Crop Candidate,18880
Grouping — Verify Source Definition,8970
Aggregate Total — Exclude from Individual-Crop Analysis,3506


In [22]:
# Save the prepared APY dataset without overwriting the source.

output_path = (
    project_root
    / "data"
    / "processed"
    / "historical_food_grains_apy_cleaned_flagged.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

apy_clean_df.to_csv(output_path, index=False)

print("Saved dataset:", output_path)
print("File exists:", output_path.exists())
print("Saved shape:", pd.read_csv(output_path).shape)

Saved dataset: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\historical_food_grains_apy_cleaned_flagged.csv
File exists: True
Saved shape: (31356, 89)


## Cleaning Summary and Decisions

### Completed
- Validated the dataset structure, column names, data types, and record uniqueness.
- Investigated missing values and historical measurement coverage.
- Examined zero values and inconsistencies between Area, Production, and Yield.
- Identified extreme Yield observations for further investigation.
- Created a provisional crop classification to distinguish aggregate totals, groupings, and individual-crop candidates.
- Added year-wise data-quality flags without modifying the original measurements.
- Saved the prepared dataset to `data/processed/historical_food_grains_apy_cleaned_flagged.csv`.

### Validation Results
- Records retained: 31,356
- Columns in the prepared dataset: 89
- Duplicate rows: 0
- Missing crop classifications: 0

### Cleaning Decisions
- Missing values were preserved rather than automatically replaced with zero.
- Suspicious Area–Production–Yield combinations were flagged rather than deleted or corrected without evidence.
- Negative Area and Production values were not found.
- Aggregate totals were retained in the dataset but marked for exclusion from individual-crop analysis.
- Crop groupings remain provisional until their definitions are verified against the source documentation.

### Limitations and Next Steps
The dataset has been quality-checked and flagged, but suspicious measurements have not all been independently verified or corrected. Yield outliers and crop-category definitions require further investigation. Subsequent preprocessing will reshape the annual measurements into a long-format dataset and establish analysis-specific inclusion rules.

The original source dataset has been preserved separately.
